# ADMM-Guided Hybrid UC — Colab GPU Launcher
This notebook does not contain a separate copy of the algorithm. It clones the project, installs the pinned Qamomile/CUDA-Q environment, selects the NVIDIA target, and starts the same FastAPI backend used by the frontend.


## 1. Enable GPU
Colab: **Runtime → Change runtime type → T4 GPU**.


In [ ]:
!nvidia-smi


## 2. Clone the project
Replace the repository URL, or upload the ZIP and unzip it under `/content`.


In [ ]:
REPO_URL = "https://github.com/YOUR_ACCOUNT/YOUR_REPOSITORY.git"
PROJECT_DIR = "/content/Quantathon-demo-2-true"
!rm -rf {PROJECT_DIR}
!git clone {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}


## 3. Install pinned dependencies
After installing CUDA-Q, Colab may request a runtime restart. Restart once, then continue from the next cell.


In [ ]:
!python -m pip install --upgrade pip
!python -m pip install -r backend/requirements-quantum-colab.txt


## 4. Select CUDA-Q NVIDIA GPU


In [ ]:
import os
import cudaq
os.environ["CUDAQ_TARGET"] = "nvidia"
os.environ["REQUIRE_CUDAQ"] = "1"  # Never silently use the NumPy fallback in the official GPU run.
cudaq.set_target("nvidia")
print("CUDA-Q target:", cudaq.get_target())


## 5. Verify the real Qamomile → CUDA-Q path
This smoke test has fallback disabled, so it fails immediately if CUDA-Q cannot execute on the selected target.


In [ ]:
%cd {PROJECT_DIR}
!PYTHONPATH=backend python backend/scripts/quantum_smoke.py


## 6. Run backend tests


In [ ]:
%cd {PROJECT_DIR}
!PYTHONPATH=backend pytest -q backend/tests


## 7. Start FastAPI
The backend now executes Qamomile → CUDA-Q with the NVIDIA target whenever the frontend calls `/api/runs`.


In [ ]:
%cd {PROJECT_DIR}/backend
!pkill -f "uvicorn app.main:app" || true
!nohup uvicorn app.main:app --host 0.0.0.0 --port 8000 > /tmp/hquc-backend.log 2>&1 &
import time, requests
time.sleep(3)
print(requests.get("http://127.0.0.1:8000/api/health").json())


## 8. Connect the frontend
Expose port 8000 with your preferred Colab tunnel. Set the resulting URL in `frontend/.env.local` as `VITE_API_BASE_URL=https://.../api`, then run the frontend locally with `npm install && npm run dev`.


In [ ]:
!tail -n 80 /tmp/hquc-backend.log
